# Bankroll Simulation: Score Margin Filter

Tests whether filtering by score margin (current_score - threshold) improves compounding returns.
All margin buckets except one showed positive mean P&L in the fragility analysis, so filtering
removes +EV trades. The question is whether the improved win rate compounds better than the
extra opportunities.

Sweeps:
1. **Ceiling sweep**: margin ≤ X for various X
2. **Band sweep**: margin in [A, B] for various combinations
3. **Cross-sweep**: min_edge × margin ceiling

In [ ]:
import sys, os, io, glob, contextlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from edge import compute_edge
from critic_model import (
    build_critic_profiles, build_kde_lambda_model,
    default_training_slugs, estimate_lambda, estimate_p_fresh,
)

PRICE_DIR = ROOT / "rt-price-histories"
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
reviews_df = pd.read_csv(ROOT / "reviews.csv")
reviews_df["estimated_timestamp"] = pd.to_datetime(
    reviews_df["estimated_timestamp"], format="ISO8601", utc=True
)
movies_df = pd.read_csv(ROOT / "movies_index.csv")
movies_df["Bet Close Date"] = pd.to_datetime(movies_df["Bet Close Date"], utc=True)

slugs_with_prices = sorted([
    d.name for d in PRICE_DIR.iterdir()
    if d.is_dir() and list(d.glob("*hour*"))
])
movies_bt = movies_df[movies_df["Slug"].isin(slugs_with_prices)].copy()
movies_bt = movies_bt.dropna(subset=["Bet Close Date"]).sort_values("Bet Close Date")
print(f"Reviews: {len(reviews_df):,} rows, {reviews_df['movie_slug'].nunique()} movies")
print(f"Movies with price data: {len(movies_bt)}")

## Backtest (daily snapshots, ~4 min)

In [ ]:
def load_hourly_prices(slug):
    csv_files = list((PRICE_DIR / slug).glob("*hour*"))
    if not csv_files:
        return None
    df = pd.read_csv(csv_files[0])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    thresh_cols = [c for c in df.columns if c.startswith("Above ")]
    df[thresh_cols] = df[thresh_cols].ffill()
    return df


def get_resolution(price_df):
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    resolution = {}
    for col in thresh_cols:
        thresh = int(col.split()[-1])
        last_valid = price_df[col].dropna()
        if last_valid.empty:
            resolution[thresh] = None
            continue
        terminal = last_valid.iloc[-1]
        if terminal >= 90:
            resolution[thresh] = True
        elif terminal <= 10:
            resolution[thresh] = False
        else:
            resolution[thresh] = None
    return resolution


def precompute_review_states(slug, reviews_df, bet_close):
    movie_reviews = reviews_df[reviews_df["movie_slug"] == slug].copy()
    movie_reviews = movie_reviews[movie_reviews["estimated_timestamp"] <= bet_close]
    movie_reviews = movie_reviews.sort_values("estimated_timestamp").reset_index(drop=True)
    if movie_reviews.empty:
        return [], None
    states = []
    critics = set()
    fresh = 0
    total = 0
    for _, row in movie_reviews.iterrows():
        critics = critics | {row["reviewer_name"]}
        total += 1
        if row["tomatometer_sentiment"] == "positive":
            fresh += 1
        states.append({
            "timestamp": row["estimated_timestamp"],
            "observed_critics": frozenset(critics),
            "fresh_count": fresh,
            "total_count": total,
        })
    return states, movie_reviews["estimated_timestamp"].iloc[0]


def get_review_state_at(states, snapshot_time):
    if not states or snapshot_time < states[0]["timestamp"]:
        return set(), 0, 0
    lo, hi = 0, len(states) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if states[mid]["timestamp"] <= snapshot_time:
            lo = mid
        else:
            hi = mid - 1
    s = states[lo]
    return set(s["observed_critics"]), s["fresh_count"], s["total_count"]

In [ ]:
def backtest_movie(slug, reviews_df, movies_df, every_n_hours=1):
    row = movies_df[movies_df["Slug"] == slug].iloc[0]
    bet_close_date = row["Bet Close Date"]
    price_df = load_hourly_prices(slug)
    if price_df is None or price_df.empty:
        return []
    market_close_time = price_df["timestamp"].iloc[-1]
    resolution = get_resolution(price_df)
    training_slugs = default_training_slugs(
        movies_df, exclude_slug=slug, before_date=bet_close_date
    )
    if len(training_slugs) < 5:
        return []
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        profiles = build_critic_profiles(reviews_df, movies_df, training_slugs)
        model = build_kde_lambda_model(profiles)
    review_states, first_review_ts = precompute_review_states(slug, reviews_df, market_close_time)
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    records = []
    cached_critics, cached_fresh, cached_total = set(), 0, 0
    cached_lambda, cached_p_fresh = None, None
    last_kept_ts = None
    for i in range(len(price_df)):
        snap_row = price_df.iloc[i]
        snapshot_time = snap_row["timestamp"]
        if every_n_hours > 1 and last_kept_ts is not None:
            hours_since = (snapshot_time - last_kept_ts).total_seconds() / 3600
            if hours_since < every_n_hours:
                continue
        last_kept_ts = snapshot_time
        hours_to_close = (market_close_time - snapshot_time).total_seconds() / 3600
        if hours_to_close <= 0:
            continue
        days_before_close = hours_to_close / 24
        observed_critics, fresh_count, total_count = get_review_state_at(
            review_states, snapshot_time
        )
        state_changed = (total_count != cached_total)
        if state_changed or cached_lambda is None:
            cached_critics = observed_critics
            cached_fresh = fresh_count
            cached_total = total_count
            first_review_dbc = None
            if first_review_ts is not None and total_count > 0:
                first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
            cached_lambda = estimate_lambda(
                model, days_before_close, hours_to_close,
                observed_critics, observed_count=total_count,
                first_review_dbc=first_review_dbc,
            )
            cached_p_fresh = estimate_p_fresh(
                profiles, observed_critics, fresh_count, total_count,
            )
        else:
            first_review_dbc = None
            if first_review_ts is not None and cached_total > 0:
                first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
            cached_lambda = estimate_lambda(
                model, days_before_close, hours_to_close,
                cached_critics, observed_count=cached_total,
                first_review_dbc=first_review_dbc,
            )
        for col in thresh_cols:
            thresh = int(col.split()[-1])
            market_price = snap_row[col]
            if pd.isna(market_price):
                continue
            resolved = resolution.get(thresh)
            if resolved is None:
                continue
            try:
                result = compute_edge(
                    threshold=thresh, market_price=market_price,
                    fresh_count=cached_fresh, total_count=cached_total,
                    hours_to_close=hours_to_close, lambda_rate=cached_lambda,
                    p_fresh=cached_p_fresh,
                )
            except (ValueError, Exception):
                continue
            current_score = (cached_fresh / cached_total * 100) if cached_total > 0 else None
            records.append({
                "slug": slug, "snapshot_time": snapshot_time,
                "hours_to_close": hours_to_close, "threshold": thresh,
                "market_price": market_price, "model_p_yes": result["p_yes"],
                "edge_cents": result["edge_cents"], "resolved_yes": resolved,
                "lambda_rate": cached_lambda, "p_fresh": cached_p_fresh,
                "fresh_count": cached_fresh, "total_count": cached_total,
                "expected_reviews": result["expected_reviews"],
                "current_score": current_score,
            })
    return records

In [ ]:
import time

EVERY_N_HOURS = 24
all_records = []
slugs = movies_bt["Slug"].tolist()
skipped = []
t0 = time.time()

for idx, slug in enumerate(slugs):
    elapsed = time.time() - t0
    rate = (idx / elapsed) if elapsed > 0 and idx > 0 else 0
    eta = (len(slugs) - idx) / rate if rate > 0 else 0
    print(f"\r[{idx+1}/{len(slugs)}] {slug:<40s} ({elapsed:.0f}s, ~{eta:.0f}s left)", end="", flush=True)
    try:
        records = backtest_movie(slug, reviews_df, movies_df, every_n_hours=EVERY_N_HOURS)
        all_records.extend(records)
    except Exception as e:
        skipped.append((slug, str(e)))

print(f"\nDone in {time.time()-t0:.0f}s. {len(all_records):,} evaluations, {len(slugs)-len(skipped)} movies.")

In [ ]:
trades = pd.DataFrame(all_records)
trades["direction"] = np.where(trades["edge_cents"] >= 0, "Yes", "No")
trades["abs_edge"] = trades["edge_cents"].abs()
trades["pnl"] = np.where(
    trades["direction"] == "Yes",
    np.where(trades["resolved_yes"], 100 - trades["market_price"], -trades["market_price"]),
    np.where(trades["resolved_yes"], -(100 - trades["market_price"]), trades["market_price"]),
)
trades["score_margin"] = trades["current_score"] - trades["threshold"]
print(f"{len(trades):,} evaluations, {trades['slug'].nunique()} movies")

## Bankroll simulation

In [ ]:
def simulate_bankroll(trades_df, min_edge, bankroll_frac,
                      margin_floor=None, margin_ceil=None,
                      start_bankroll=100000.0):
    """Simulate bankroll growth for a No-only strategy.
    
    Treats all positions within a movie as perfectly correlated (worst case).
    Sizes each movie's total risk as bankroll_frac of current bankroll.
    
    Args:
        margin_floor: minimum score_margin to include (None = no floor)
        margin_ceil: maximum score_margin to include (None = no ceiling)
    """
    ACTION_WINDOW = (24, 120)
    mask = (
        (trades_df["direction"] == "No") &
        (trades_df["abs_edge"] >= min_edge) &
        (trades_df["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades_df["hours_to_close"] <= ACTION_WINDOW[1])
    )
    if margin_floor is not None:
        mask &= (trades_df["score_margin"] >= margin_floor)
    if margin_ceil is not None:
        mask &= (trades_df["score_margin"] <= margin_ceil)
    
    no_trades = trades_df[mask].sort_values("snapshot_time")
    positions = no_trades.groupby(["slug", "threshold"]).first().reset_index()
    positions["entry_cost"] = 100 - positions["market_price"]
    positions["pos_pnl"] = np.where(
        ~positions["resolved_yes"],
        positions["market_price"],
        -positions["entry_cost"],
    )
    
    movie_results = positions.groupby("slug").agg(
        first_entry=("snapshot_time", "min"),
        total_pnl=("pos_pnl", "sum"),
        total_cost=("entry_cost", "sum"),
        n_positions=("pos_pnl", "count"),
    ).sort_values("first_entry").reset_index()
    
    bankroll = start_bankroll
    trajectory = [{"movie": "START", "bankroll": bankroll,
                   "time": movie_results["first_entry"].min() if len(movie_results) > 0 else pd.NaT}]
    
    for _, movie in movie_results.iterrows():
        risk_budget = bankroll * bankroll_frac
        contracts_scale = risk_budget / movie["total_cost"] if movie["total_cost"] > 0 else 0
        movie_pnl = movie["total_pnl"] * contracts_scale
        bankroll += movie_pnl
        if bankroll <= 0:
            bankroll = 0
            trajectory.append({"movie": movie["slug"], "bankroll": 0, "time": movie["first_entry"]})
            break
        trajectory.append({"movie": movie["slug"], "bankroll": bankroll, "time": movie["first_entry"]})
    
    return pd.DataFrame(trajectory), movie_results

### Ceiling sweep: margin ≤ X

In [ ]:
START_BANKROLL = 100000  # $1,000 in cents
MIN_EDGE = 10
BANKROLL_FRAC = 0.10

ceilings = [None, 10, 5, 2, 0, -2, -5]
ceil_results = []

for ceil in ceilings:
    traj, mr = simulate_bankroll(trades, MIN_EDGE, BANKROLL_FRAC,
                                 margin_ceil=ceil, start_bankroll=START_BANKROLL)
    final = traj["bankroll"].iloc[-1]
    n_movies = len(mr)
    n_pos = mr["n_positions"].sum()
    win_movies = (mr["total_pnl"] > 0).sum()
    ceil_results.append({
        "margin_ceil": f"<= {ceil}" if ceil is not None else "No filter",
        "movies": n_movies,
        "positions": n_pos,
        "movie_win_rate": win_movies / n_movies if n_movies > 0 else 0,
        "final_$": final / 100,
        "multiplier": final / START_BANKROLL,
    })

ceil_df = pd.DataFrame(ceil_results)
print(f"=== Ceiling sweep (min_edge={MIN_EDGE}c, risk={BANKROLL_FRAC:.0%}/movie, start=${START_BANKROLL/100:,.0f}) ===")
print(ceil_df.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for ceil in [None, 5, 2, 0, -2, -5]:
    label = f"margin <= {ceil}" if ceil is not None else "No filter"
    traj, _ = simulate_bankroll(trades, MIN_EDGE, BANKROLL_FRAC,
                                margin_ceil=ceil, start_bankroll=START_BANKROLL)
    ax.plot(traj["time"], traj["bankroll"] / 100, label=label, linewidth=1.2)

ax.axhline(START_BANKROLL / 100, color="k", linewidth=0.3, linestyle="--")
ax.set_xlabel("Time")
ax.set_ylabel("Bankroll ($)")
ax.set_title(f"Ceiling sweep (min_edge={MIN_EDGE}c, risk={BANKROLL_FRAC:.0%})")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Band sweep: margin in [A, B]

In [ ]:
bands = [
    (None, None, "No filter"),
    (None, 0, "<= 0"),
    (None, 2, "<= +2"),
    (-10, 0, "-10 to 0"),
    (-5, 0, "-5 to 0"),
    (-5, 2, "-5 to +2"),
    (-5, 5, "-5 to +5"),
    (-10, -2, "-10 to -2"),
    (-10, 2, "-10 to +2"),
    (-15, 0, "-15 to 0"),
    (0, 5, "0 to +5 (above only)"),
]

band_results = []
for floor, ceil, label in bands:
    traj, mr = simulate_bankroll(trades, MIN_EDGE, BANKROLL_FRAC,
                                 margin_floor=floor, margin_ceil=ceil,
                                 start_bankroll=START_BANKROLL)
    final = traj["bankroll"].iloc[-1]
    n_movies = len(mr)
    n_pos = mr["n_positions"].sum()
    win_movies = (mr["total_pnl"] > 0).sum()
    band_results.append({
        "band": label,
        "movies": n_movies,
        "positions": n_pos,
        "movie_win_rate": win_movies / n_movies if n_movies > 0 else 0,
        "final_$": final / 100,
        "multiplier": final / START_BANKROLL,
    })

band_df = pd.DataFrame(band_results)
print(f"=== Band sweep (min_edge={MIN_EDGE}c, risk={BANKROLL_FRAC:.0%}/movie, start=${START_BANKROLL/100:,.0f}) ===")
print(band_df.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

key_bands = [
    (None, None, "No filter"),
    (None, 0, "<= 0"),
    (-5, 0, "-5 to 0"),
    (-10, 0, "-10 to 0"),
    (-5, 2, "-5 to +2"),
    (0, 5, "0 to +5 only"),
]

for floor, ceil, label in key_bands:
    traj, _ = simulate_bankroll(trades, MIN_EDGE, BANKROLL_FRAC,
                                margin_floor=floor, margin_ceil=ceil,
                                start_bankroll=START_BANKROLL)
    ax.plot(traj["time"], traj["bankroll"] / 100, label=label, linewidth=1.2)

ax.axhline(START_BANKROLL / 100, color="k", linewidth=0.3, linestyle="--")
ax.set_xlabel("Time")
ax.set_ylabel("Bankroll ($)")
ax.set_title(f"Band sweep (min_edge={MIN_EDGE}c, risk={BANKROLL_FRAC:.0%})")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Cross-sweep: min_edge × margin ceiling

In [ ]:
rows = []
for me in [5, 10, 15, 20]:
    for ceil in [None, 5, 2, 0, -2]:
        traj, mr = simulate_bankroll(trades, me, BANKROLL_FRAC,
                                     margin_ceil=ceil, start_bankroll=START_BANKROLL)
        final = traj["bankroll"].iloc[-1]
        n_pos = mr["n_positions"].sum()
        rows.append({
            "min_edge": me,
            "margin_ceil": f"<= {ceil}" if ceil is not None else "None",
            "positions": n_pos,
            "movies": len(mr),
            "final_$": final / 100,
            "multiplier": final / START_BANKROLL,
        })

cross_df = pd.DataFrame(rows)
print(f"=== Cross-sweep: min_edge x margin ceiling (risk={BANKROLL_FRAC:.0%}, start=${START_BANKROLL/100:,.0f}) ===")
print(cross_df.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
print(f"\nBest overall: {cross_df.loc[cross_df['multiplier'].idxmax()].to_dict()}")

In [ ]:
# Heatmap: multiplier by min_edge x margin_ceil
pivot = cross_df.pivot(index="min_edge", columns="margin_ceil", values="multiplier")
# Reorder columns
col_order = [c for c in ["None", "<= 5", "<= 2", "<= 0", "<= -2"] if c in pivot.columns]
pivot = pivot[col_order]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{x}c" for x in pivot.index])
ax.set_xlabel("Margin Ceiling")
ax.set_ylabel("Min Edge")
ax.set_title("Bankroll Multiplier: min_edge x margin ceiling")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.1f}x", ha="center", va="center", fontsize=11,
                fontweight="bold" if val == pivot.values.max() else "normal")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()